# Kaggle: ResNet18 + MTCNN Deepfake Training

Same pipeline as the local BTech project:

1. ImageNet-pretrained **ResNet18**
2. Head: `Dropout(0.5)` + `Linear(512 → 1)` logit
3. **BCEWithLogitsLoss** + **Adam** (`weight_decay=1e-4`) + **ReduceLROnPlateau**
4. **MTCNN** (`min_face_size=20`, `thresholds=[0.6,0.7,0.7]`), largest face, **padding=0.2**, **224×224**, ImageNet normalize

**Before running**

1. Set notebook accelerator to **GPU**.
2. Add your dataset. Folders must be `train/real`, `train/fake`, `val/real`, `val/fake`.
3. Edit `DATASET_DIR` in the config cell.
4. After training, download `/kaggle/working/detector.pt` and copy it to `backend/data/detector.pt` on your laptop.

In [ ]:
!pip install -q facenet-pytorch

In [ ]:
from pathlib import Path

# CHANGE THIS to the folder that contains train/ and val/
DATASET_DIR = Path("/kaggle/input/YOUR_DATASET_NAME")

OUTPUT_DIR = Path("/kaggle/working")
MODEL_PATH = OUTPUT_DIR / "detector.pt"
RESULTS_DIR = OUTPUT_DIR / "training_plots"
EVAL_JSON_PATH = OUTPUT_DIR / "evaluation_report.json"

EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 42
IMAGE_SIZE = 224
EXTRACT_FACE = True
FACE_PADDING = 0.2
NUM_WORKERS = 2

print("Edit DATASET_DIR if this path is wrong:", DATASET_DIR)
print("GPU expected. cuda available will be printed after imports.")

In [ ]:
import json
import math
import os
import random
from typing import Callable, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from facenet_pytorch import MTCNN
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def resolve_dataset_dir(configured: Path) -> Path:
    if (configured / "train").is_dir() and (configured / "val").is_dir():
        return configured
    if configured.exists():
        for child in configured.iterdir():
            if child.is_dir() and (child / "train").is_dir() and (child / "val").is_dir():
                print("Using nested dataset folder:", child)
                return child
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for ds in sorted(kaggle_input.iterdir()):
            if not ds.is_dir():
                continue
            if (ds / "train").is_dir() and (ds / "val").is_dir():
                print("Auto-detected:", ds)
                return ds
            for child in ds.iterdir():
                if child.is_dir() and (child / "train").is_dir() and (child / "val").is_dir():
                    print("Auto-detected nested:", child)
                    return child
    raise FileNotFoundError("Could not find train/ and val/. Set DATASET_DIR correctly.")


class ResNet18Detector(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_features, 1))

    def forward(self, x):
        return self.backbone(x)


class RandomGamma:
    def __init__(self, gamma_range=(0.8, 1.2), p: float = 0.5):
        self.gamma_range = gamma_range
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() > self.p:
            return img
        gamma = float(np.random.uniform(*self.gamma_range))
        arr = np.clip((np.asarray(img).astype(np.float32) / 255.0) ** gamma, 0.0, 1.0)
        return Image.fromarray((arr * 255.0).astype(np.uint8))


def get_train_transforms(image_size: int = 224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.1, saturation=0.1, hue=0.05),
        RandomGamma(),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_val_transforms(image_size: int = 224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None, extract_face=True, device=None, padding=0.2):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.extract_face = extract_face
        self.padding = padding
        self.device = device or torch.device("cpu")
        self.samples = []
        for label_name, label in (("real", 0.0), ("fake", 1.0)):
            folder = self.root_dir / label_name
            if folder.is_dir():
                for p in folder.iterdir():
                    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS:
                        self.samples.append((p, label))
        self.mtcnn = None
        if extract_face:
            self.mtcnn = MTCNN(
                image_size=224, margin=0, min_face_size=20, thresholds=[0.6, 0.7, 0.7],
                keep_all=True, post_process=False, device=self.device,
            )

    def __len__(self):
        return len(self.samples)

    def _extract_padded_face(self, pil_image):
        if self.mtcnn is None:
            return pil_image
        try:
            boxes, _ = self.mtcnn.detect(pil_image)
            if boxes is None or len(boxes) == 0:
                return pil_image
            largest, max_area = None, -1.0
            for box in boxes:
                x1, y1, x2, y2 = box
                area = max(0.0, float(x2 - x1)) * max(0.0, float(y2 - y1))
                if area > max_area:
                    max_area, largest = area, box
            if largest is None:
                return pil_image
            img_w, img_h = pil_image.size
            x1, y1, x2, y2 = largest
            pad_x = int(math.floor((x2 - x1) * self.padding))
            pad_y = int(math.floor((y2 - y1) * self.padding))
            nx1 = max(0, int(round(x1 - pad_x)))
            ny1 = max(0, int(round(y1 - pad_y)))
            nx2 = min(img_w, int(round(x2 + pad_x)))
            ny2 = min(img_h, int(round(y2 + pad_y)))
            if nx2 > nx1 and ny2 > ny1:
                return pil_image.crop((nx1, ny1, nx2, ny2))
            return pil_image
        except Exception:
            return pil_image

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.extract_face:
                img = self._extract_padded_face(img)
            tensor = self.transform(img) if self.transform is not None else transforms.ToTensor()(img)
        return tensor, torch.tensor([label], dtype=torch.float32)


def set_seed(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print("Helpers loaded.")

In [ ]:
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_root = resolve_dataset_dir(DATASET_DIR)

train_dataset = DeepfakeDataset(
    dataset_root / "train", get_train_transforms(IMAGE_SIZE), EXTRACT_FACE, device, FACE_PADDING
)
val_dataset = DeepfakeDataset(
    dataset_root / "val", get_val_transforms(IMAGE_SIZE), EXTRACT_FACE, device, FACE_PADDING
)
print("Train:", len(train_dataset), "Val:", len(val_dataset), "Device:", device)
assert len(train_dataset) > 0 and len(val_dataset) > 0, "Empty dataset folders."

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=device.type=="cuda")
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type=="cuda")

model = ResNet18Detector(pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)

history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * images.size(0)
    train_loss = running / len(train_dataset)

    model.eval()
    running, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            running += criterion(logits, labels).item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5).float() == labels).sum().item()
    val_loss = running / len(val_dataset)
    val_acc = (correct / len(val_dataset)) * 100.0
    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch:02d}/{EPOCHS}  train {train_loss:.4f}  val {val_loss:.4f}  acc {val_acc:.2f}%  lr {optimizer.param_groups[0]['lr']:.6f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": "resnet18",
            "image_size": IMAGE_SIZE,
            "val_acc": best_val_acc,
            "val_loss": val_loss,
            "epoch": epoch,
        }, str(MODEL_PATH))
        print("  saved best ->", MODEL_PATH)

print("Best val acc:", round(best_val_acc, 2))

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
epochs = list(range(1, len(history["train_loss"]) + 1))

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], "b-o"); plt.title("Training Loss vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("BCE Loss"); plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "training_loss.png", dpi=200); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["val_loss"], "r-o"); plt.title("Validation Loss vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("BCE Loss"); plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "validation_loss.png", dpi=200); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["val_acc"], "g-o"); plt.title("Validation Accuracy vs Epoch"); plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.ylim(0, 105); plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "validation_accuracy.png", dpi=200); plt.show()

In [ ]:
checkpoint = torch.load(str(MODEL_PATH), map_location=device)
eval_model = ResNet18Detector(pretrained=False)
eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.to(device).eval()

all_labels, all_preds = [], []
total_loss = 0.0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        logits = eval_model(images)
        total_loss += nn.BCEWithLogitsLoss()(logits, labels).item() * images.size(0)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        all_preds.extend((probs >= 0.5).astype(int).tolist())
        all_labels.extend(labels.cpu().numpy().flatten().astype(int).tolist())

cm = confusion_matrix(all_labels, all_preds).tolist()
report = {
    "samples_evaluated": len(val_dataset),
    "validation_loss": round(total_loss / len(val_dataset), 4),
    "accuracy_percent": round(accuracy_score(all_labels, all_preds) * 100.0, 2),
    "precision_percent": round(precision_score(all_labels, all_preds, zero_division=0) * 100.0, 2),
    "recall_percent": round(recall_score(all_labels, all_preds, zero_division=0) * 100.0, 2),
    "f1_score_percent": round(f1_score(all_labels, all_preds, zero_division=0) * 100.0, 2),
    "confusion_matrix": cm,
}
print(json.dumps(report, indent=2))
EVAL_JSON_PATH.write_text(json.dumps(report, indent=4))
print("Download these from /kaggle/working:")
print(" -", MODEL_PATH)
print(" -", EVAL_JSON_PATH)
print("Then copy detector.pt to backend/data/detector.pt on your laptop.")